In [61]:
from DataLoader import DataLoader
import pandas as pd

In [63]:
class StrategyEngine:
    def __init__(self, data):
        if data is None or data.empty:
            raise ValueError(
                "Input data is empty. Please fetch data before strategy execution."
            )

        self.data = data.copy()

    def _get_close_series(self):

        if "Close" in self.data.columns:
            close = self.data["Close"]
        else:
            close_cols = []

            for col in self.data.columns:
                if "Close" in str(col):
                    close_cols.append(col)

            if len(close_cols) == 0:
                raise KeyError("Close price column not found in input data.")

            close = self.data[close_cols[0]]

        # Convert to Series if needed
        if hasattr(close, "columns"):
            close = close.iloc[:, 0]

        return close.astype(float)

    def calculate_moving_average(self, prices, window):
        moving_averages = []

        for i in range(len(prices)):
            if i < window - 1:
                moving_averages.append(None)
            else:
                total = 0

                for j in range(i - window + 1, i + 1):
                    total += prices.iloc[j]

                average = total / window
                moving_averages.append(average)

        return moving_averages

    def moving_average_crossover(self, short_window=20, long_window=50):
        if short_window <= 0 or long_window <= 0:
            raise ValueError("Window sizes must be positive integers.")

        if short_window >= long_window:
            raise ValueError("short_window must be less than long_window.")

        close = self._get_close_series()

        strategy_df = pd.DataFrame(index=self.data.index)

        strategy_df["Close"] = close

        # Manual moving average calculation
        strategy_df["SMA_Short"] = self.calculate_moving_average(
            close,
            short_window
        )

        strategy_df["SMA_Long"] = self.calculate_moving_average(
            close,
            long_window
        )

        # Position generation
        positions = []
        for i in range(len(strategy_df)):
            short_ma = strategy_df["SMA_Short"].iloc[i]
            long_ma = strategy_df["SMA_Long"].iloc[i]

            if pd.isna(short_ma) or pd.isna(long_ma):
                positions.append(0)
            elif short_ma > long_ma:
                positions.append(1)
            else:
                positions.append(0)

        strategy_df["Position"] = positions

        # Manual signal generation
        signals = [0]
        for i in range(1, len(positions)):
            signal = positions[i] - positions[i - 1]
            signals.append(signal)

        strategy_df["Signal"] = signals

        return strategy_df

    def parameter_tuning(self, short_windows, long_windows):
        close = self._get_close_series()
        records = []
        # Daily returns
        daily_returns = []

        for i in range(len(close)):
            if i == 0:
                daily_returns.append(0)
            else:
                previous_price = close.iloc[i - 1]
                current_price = close.iloc[i]
                daily_return = ( current_price - previous_price) / previous_price
                daily_returns.append(daily_return)
        
        for short_window in short_windows:
            for long_window in long_windows:
                if short_window >= long_window:
                    continue
                strat = self.moving_average_crossover( short_window=short_window,long_window=long_window)
                positions = strat["Position"].tolist()

                strategy_returns = []
                for i in range(len(daily_returns)):
                    if i == 0:
                        strategy_returns.append(0)
                    else:
                        previous_position = positions[i - 1]
                        strategy_return = (
                            daily_returns[i] * previous_position
                        )
                        strategy_returns.append(strategy_return)

                total_return = 1

                for ret in strategy_returns:
                    total_return *= (1 + ret)

                total_return -= 1

                records.append({
                    "short_window": short_window,
                    "long_window": long_window,
                    "total_return": total_return
                })

        if len(records) == 0:
            raise ValueError(
                "No valid short/long window combinations found."
            )

        tuning_df = pd.DataFrame(records)

        tuning_df = tuning_df.sort_values("total_return",ascending=False ).reset_index(drop=True)

        return tuning_df

In [65]:
loader = DataLoader("AAPL", "2022-04-01", "2026-04-01")
apple = loader.get_data()
apple = loader.clean_data(apple)

[*********************100%***********************]  1 of 1 completed

Variables are successfully initialised


In [67]:
apple.head()

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2022-04-01,170.786209,171.344693,168.464125,170.511871,78751300
2022-04-04,174.832779,174.881771,170.913640,171.041017,76468400
2022-04-05,171.521088,174.695595,170.894026,173.911764,73401800
2022-04-06,168.356354,170.119969,166.690723,168.875638,89058800
2022-04-07,168.660080,169.855418,166.416380,167.699895,77594700


In [69]:
strategy_engine = StrategyEngine(apple)
strategy_output = strategy_engine.moving_average_crossover(short_window=20, long_window=50)

# Show the latest rows to verify signals are being generated
strategy_output.tail(10)

,Close,SMA_Short,SMA_Long,Position,Signal
Date,,,,,
2026-03-18,249.940002,261.398499,261.532903,0,-1
2026-03-19,248.960007,260.817500,261.269809,0,0
2026-03-20,247.990005,259.988000,261.027877,0,0
2026-03-23,251.490005,259.253501,260.881720,0,0
2026-03-24,251.639999,258.228500,260.731969,0,0
2026-03-25,252.619995,257.148000,260.584235,0,0
2026-03-26,252.889999,256.144999,260.425916,0,0
2026-03-27,248.800003,255.375999,260.207576,0,0
2026-03-30,246.630005,254.471500,259.980804,0,0


In [71]:
# Buy and sell counts for quick validation
buy_signals = int((strategy_output["Signal"] == 1).sum())
sell_signals = int((strategy_output["Signal"] == -1).sum())

print(f"Buy signals: {buy_signals}")
print(f"Sell signals: {sell_signals}")

Buy signals: 12
Sell signals: 12


In [73]:
# Parameter tuning across a small grid
short_window_candidates = [5, 10, 15, 20]
long_window_candidates = [30, 40, 50, 60]

tuning_results = strategy_engine.parameter_tuning(short_window_candidates, long_window_candidates)

print("Top 5 parameter combinations by total return:")
tuning_results.head(5)

Top 5 parameter combinations by total return:


,short_window,long_window,total_return
0,20,60,0.647250
1,15,50,0.539726
2,15,60,0.534881
3,10,50,0.516731
4,10,60,0.492700
